# Parte 01 · Carga y limpieza de datos — Dataset: Calificaciones

**Proyecto final · Minería de Datos 1 · Grupo #2**
Colegio Universitario de Cartago — Tecnologías de la Información

**Dataset:** `student_grade_ses_dataset.csv`, guardado como `data/raw/calificaciones.csv`. Contiene datos de estudiantes (familia, situación económica, hábitos de estudio y notas).

**¿Qué hacemos en esta parte?**
Cargamos los datos y los revisamos para ver qué tan "limpios" vienen. Buscamos:
- valores vacíos,
- filas repetidas,
- datos que no tienen sentido.

Después corregimos lo necesario y guardamos el resultado en `data/processed/calificaciones_limpio.csv`. Ese es el archivo que usan las partes 02 a 05.

**Contenido**
1. Importación de librerías
2. Carga de datos
3. Dimensiones y vista general
4. Tipos de variables
5. Valores nulos
6. Registros duplicados
7. Inconsistencias
8. Valores atípicos
9. Limpieza de datos
10. Revisión final y exportación
11. Conclusiones

## 1. Importación de librerías

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Ocultamos advertencias y configuramos cómo se muestran las tablas
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)       # mostrar todas las columnas
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)  # números con 2 decimales

print("pandas:", pd.__version__, "| numpy:", np.__version__)

pandas: 2.3.3 | numpy: 2.4.3


## 2. Carga de datos

Usamos rutas relativas (con `../../`) para que el notebook funcione en la computadora de cualquier integrante del grupo.

In [2]:
RUTA_RAW = Path("../../data/raw/calificaciones.csv")
RUTA_PROCESSED = Path("../../data/processed/calificaciones_limpio.csv")

df_raw = pd.read_csv(RUTA_RAW, encoding="utf-8")
df = df_raw.copy()   # trabajamos con una copia para no tocar los datos originales
df.head()

,student_id,academic_year,survey_date,age,gender,grade_level,school_type,residence_type,family_size,num_siblings,single_parent_household,family_income_bracket,annual_family_income_inr,father_education_level,mother_education_level,father_occupation,mother_occupation,has_internet_access,has_computer_or_laptop,distance_to_school_km,scholarship_status,extra_tutoring,parental_involvement_score,study_hours_per_week,attendance_rate_pct,previous_year_grade_pct,final_grade_pct,grade_class,at_risk_flag
0,STU101529,2024-2025,2025-03-06,16,Male,12,Government,Rural,3,0,1,Middle,"403,980.00",Primary,Secondary,Clerical / Service,Homemaker,0.00,0,4.00,No Scholarship,0,4,15.00,76.80,61.70,59.40,D,0
1,STU107546,2022-2023,2025-05-02,15,Male,10,Government,Urban,3,1,0,Middle,"572,014.00",NaN,Secondary,Business / Self-Employed,Unemployed,0.00,1,0.70,No Scholarship,0,3,20.90,74.50,49.20,60.20,C,0
2,STU103737,2024-2025,2025-03-21,14,Male,9,Government,Urban,2,0,0,Low,"90,092.00",No Formal Education,Bachelor's Degree,Clerical / Service,Managerial / Executive,1.00,0,2.90,No Scholarship,0,4,20.90,94.90,56.90,58.40,D,0
3,STU108172,2023-2024,2025-03-30,13,Male,8,Government,Rural,6,4,0,Lower-Middle,"260,583.00",Higher Secondary,Secondary,Skilled Labor / Trade,Homemaker,0.00,0,7.00,No Scholarship,0,1,6.50,63.60,49.30,57.00,D,0
4,STU110633,2023-2024,2025-05-06,11,Female,6,Private-Unaided,Rural,5,3,1,Upper-Middle,"1,114,986.00",Bachelor's Degree,Higher Secondary,Professional,Clerical / Service,1.00,0,9.10,No Scholarship,0,5,13.90,86.00,79.30,88.70,B,0


## 3. Dimensiones y vista general

In [3]:
# shape nos dice cuántas filas y columnas hay
filas, columnas = df.shape
print(f"Registros (filas): {filas:,}")
print(f"Variables (columnas): {columnas}")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Registros (filas): 12,025
Variables (columnas): 29
Memoria utilizada: 11.40 MB


In [4]:
df.tail()  # últimas filas

,student_id,academic_year,survey_date,age,gender,grade_level,school_type,residence_type,family_size,num_siblings,single_parent_household,family_income_bracket,annual_family_income_inr,father_education_level,mother_education_level,father_occupation,mother_occupation,has_internet_access,has_computer_or_laptop,distance_to_school_km,scholarship_status,extra_tutoring,parental_involvement_score,study_hours_per_week,attendance_rate_pct,previous_year_grade_pct,final_grade_pct,grade_class,at_risk_flag
12020,STU104023,2022-2023,2025-05-08,14,Male,8,Government,Urban,4,1,0,Lower-Middle,"289,117.00",Secondary,Master's Degree,Unskilled Labor,Business / Self-Employed,0.00,1,4.00,Full,1,3,17.20,95.70,71.00,77.20,B,0
12021,STU107259,2022-2023,2025-04-10,12,Female,7,Government-Aided,Urban,7,4,0,Lower-Middle,"227,105.00",Secondary,Secondary,Business / Self-Employed,Business / Self-Employed,0.00,1,0.80,No Scholarship,1,3,8.80,90.50,66.80,63.10,C,0
12022,STU105200,2024-2025,2025-03-26,15,Male,9,Government-Aided,Rural,6,4,0,Low,"88,937.00",Primary,Primary,Unemployed,Managerial / Executive,0.00,0,4.10,Partial,0,2,10.10,80.70,48.50,49.90,D,0
12023,STU103775,2024-2025,2025-04-09,11,Male,7,Private-Unaided,Urban,3,1,0,Low,"70,487.00",No Formal Education,Primary,Unemployed,Unskilled Labor,0.00,1,1.40,No Scholarship,0,3,14.40,76.70,47.60,56.50,D,0
12024,STU110137,2023-2024,2025-04-17,12,Female,7,Government,Rural,4,2,1,Low,"158,198.00",Secondary,Secondary,NaN,Unemployed,0.00,1,6.30,Partial,0,2,11.10,78.40,62.10,74.60,C,0


In [5]:
df.info()  # tipo de cada columna y cuántos valores no vacíos tiene

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12025 entries, 0 to 12024
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  12025 non-null  object 
 1   academic_year               12025 non-null  object 
 2   survey_date                 12025 non-null  object 
 3   age                         12025 non-null  int64  
 4   gender                      12025 non-null  object 
 5   grade_level                 12025 non-null  int64  
 6   school_type                 12025 non-null  object 
 7   residence_type              12025 non-null  object 
 8   family_size                 12025 non-null  int64  
 9   num_siblings                12025 non-null  int64  
 10  single_parent_household     12025 non-null  int64  
 11  family_income_bracket       12025 non-null  object 
 12  annual_family_income_inr    11304 non-null  float64
 13  father_education_level      116

**¿Qué vemos?**
- El dataset tiene **12,025 filas y 29 columnas**.
- Cada fila es un estudiante. Tiene datos como su edad, su familia, el ingreso de la casa, cuántas horas estudia y sus notas.
- En `df.info()` algunas columnas tienen menos de 12,025 valores. Eso quiere decir que tienen datos vacíos.

## 4. Tipos de variables

Hacemos una tabla (diccionario de datos) con cada columna, su tipo y qué clase de dato guarda.

In [6]:
diccionario = pd.DataFrame({
    "variable": df.columns,
    "tipo_pandas": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique().values,
    "ejemplo": df.iloc[0].values,
})

# Tipo de dato según su significado (no según cómo lo lee pandas)
tipo_conceptual = {
    "student_id": "Identificador", "academic_year": "Categórica ordinal", "survey_date": "Fecha",
    "age": "Numérica discreta", "gender": "Categórica nominal", "grade_level": "Numérica discreta (ordinal)",
    "school_type": "Categórica nominal", "residence_type": "Categórica nominal",
    "family_size": "Numérica discreta", "num_siblings": "Numérica discreta",
    "single_parent_household": "Binaria (0/1)", "family_income_bracket": "Categórica ordinal",
    "annual_family_income_inr": "Numérica continua", "father_education_level": "Categórica ordinal",
    "mother_education_level": "Categórica ordinal", "father_occupation": "Categórica nominal",
    "mother_occupation": "Categórica nominal", "has_internet_access": "Binaria (0/1)",
    "has_computer_or_laptop": "Binaria (0/1)", "distance_to_school_km": "Numérica continua",
    "scholarship_status": "Categórica ordinal", "extra_tutoring": "Binaria (0/1)",
    "parental_involvement_score": "Numérica discreta (escala 1-5)", "study_hours_per_week": "Numérica continua",
    "attendance_rate_pct": "Numérica continua (%)", "previous_year_grade_pct": "Numérica continua (%)",
    "final_grade_pct": "Numérica continua (%)", "grade_class": "Categórica ordinal (A-F)",
    "at_risk_flag": "Binaria (0/1)",
}
diccionario["tipo_conceptual"] = diccionario["variable"].map(tipo_conceptual)
diccionario

,variable,tipo_pandas,valores_unicos,ejemplo,tipo_conceptual
0,student_id,object,12000,STU101529,Identificador
1,academic_year,object,4,2024-2025,Categórica ordinal
2,survey_date,object,75,2025-03-06,Fecha
3,age,int64,9,16,Numérica discreta
4,gender,object,3,Male,Categórica nominal
5,grade_level,int64,7,12,Numérica discreta (ordinal)
6,school_type,object,3,Government,Categórica nominal
7,residence_type,object,3,Rural,Categórica nominal
8,family_size,int64,9,3,Numérica discreta
9,num_siblings,int64,9,0,Numérica discreta


In [7]:
print("Cantidad de columnas por tipo de pandas:")
print(df.dtypes.value_counts().to_string())
print()
print("Cantidad de columnas por tipo de dato:")
print(diccionario["tipo_conceptual"].str.split(" ").str[0].value_counts().to_string())

Cantidad de columnas por tipo de pandas:
object     13
int64       9
float64     7

Cantidad de columnas por tipo de dato:
tipo_conceptual
Categórica       11
Numérica         11
Binaria           5
Identificador     1
Fecha             1


**¿Qué vemos?**
- Pandas lee **13 columnas como texto (`object`)**, **9 como enteros (`int64`)** y **7 como decimales (`float64`)**.
- `has_internet_access` solo puede valer 0 o 1, pero aparece como decimal. Pasa porque tiene datos vacíos y pandas los guarda como `NaN`, que es un decimal. Lo arreglamos más adelante.
- `survey_date` es una fecha, pero está guardada como texto.
- Algunas columnas de texto tienen un orden lógico, como el nivel de ingreso (Low → High) o la nota en letra (F → A). Esto le sirve a quien haga la parte 04.
- `student_id` es solo un código del estudiante. Nos sirve para buscar repetidos, pero no para el análisis.

## 5. Valores nulos

Revisamos cuántos datos vacíos hay en cada columna.

In [8]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),                          # cantidad de vacíos
    "porcentaje": (df.isna().mean() * 100).round(2),   # % de vacíos
    "tipo": df.dtypes.astype(str),
})
nulos = nulos[nulos["nulos"] > 0].sort_values("nulos", ascending=False)  # solo columnas con vacíos
print(f"Total de celdas vacías: {df.isna().sum().sum():,} de {df.size:,} ({df.isna().sum().sum() / df.size * 100:.2f}%)")
print(f"Filas con al menos un vacío: {df.isna().any(axis=1).sum():,} ({df.isna().any(axis=1).mean() * 100:.2f}%)")
nulos

Total de celdas vacías: 3,667 de 348,725 (1.05%)


Filas con al menos un vacío: 3,222 (26.79%)


,nulos,porcentaje,tipo
annual_family_income_inr,721,6.00,float64
mother_occupation,602,5.01,object
father_occupation,481,4.00,object
father_education_level,420,3.49,object
mother_education_level,361,3.00,object
previous_year_grade_pct,360,2.99,float64
attendance_rate_pct,300,2.49,float64
distance_to_school_km,241,2.00,float64
has_internet_access,181,1.51,float64


**¿Qué vemos?**
- **9 de las 29 columnas** tienen datos vacíos. La que más tiene es el ingreso familiar (`annual_family_income_inr`), con un **6%**.
- En total, **3,222 filas (26.8%)** tienen al menos un dato vacío. Si borráramos todas esas filas perderíamos más de la cuarta parte de los datos. Por eso decidimos **rellenarlos** (imputar).
- Las notas (`final_grade_pct`, `grade_class`) no tienen vacíos, y eso es bueno porque son de las columnas más importantes.

## 6. Registros duplicados

In [9]:
dup_filas = df.duplicated().sum()                               # filas idénticas
dup_ids = df["student_id"].duplicated().sum()                   # códigos de estudiante repetidos
dup_sin_id = df.drop(columns="student_id").duplicated().sum()   # mismos datos aunque cambie el código

print(f"Filas completamente duplicadas: {dup_filas}")
print(f"student_id repetidos:            {dup_ids}")
print(f"Duplicados ignorando student_id: {dup_sin_id}")

Filas completamente duplicadas: 25
student_id repetidos:            25
Duplicados ignorando student_id: 25


In [10]:
# Mostramos algunos duplicados juntos para compararlos
ids_dup = df.loc[df["student_id"].duplicated(), "student_id"]
df[df["student_id"].isin(ids_dup)].sort_values("student_id").head(6)

,student_id,academic_year,survey_date,age,gender,grade_level,school_type,residence_type,family_size,num_siblings,single_parent_household,family_income_bracket,annual_family_income_inr,father_education_level,mother_education_level,father_occupation,mother_occupation,has_internet_access,has_computer_or_laptop,distance_to_school_km,scholarship_status,extra_tutoring,parental_involvement_score,study_hours_per_week,attendance_rate_pct,previous_year_grade_pct,final_grade_pct,grade_class,at_risk_flag
6089,STU100191,2022-2023,2025-03-08,15,Female,10,Government,Semi-Urban,3,1,0,Upper-Middle,"943,780.00",Bachelor's Degree,Bachelor's Degree,Clerical / Service,Professional,1.00,1,2.10,No Scholarship,0,2,15.10,79.40,67.70,74.80,C,0
4725,STU100191,2022-2023,2025-03-08,15,Female,10,Government,Semi-Urban,3,1,0,Upper-Middle,"943,780.00",Bachelor's Degree,Bachelor's Degree,Clerical / Service,Professional,1.00,1,2.10,No Scholarship,0,2,15.10,79.40,67.70,74.80,C,0
45,STU100247,2024-2025,2025-03-10,13,Male,8,Private-Unaided,Urban,5,3,0,Low,"66,458.00",Higher Secondary,Primary,Unemployed,Unemployed,1.00,1,1.10,No Scholarship,0,3,13.20,83.80,40.40,47.00,D,0
2052,STU100247,2024-2025,2025-03-10,13,Male,8,Private-Unaided,Urban,5,3,0,Low,"66,458.00",Higher Secondary,Primary,Unemployed,Unemployed,1.00,1,1.10,No Scholarship,0,3,13.20,83.80,40.40,47.00,D,0
8776,STU100694,2022-2023,2025-03-16,11,Male,6,Government-Aided,Urban,4,2,0,Lower-Middle,NaN,Higher Secondary,Primary,Clerical / Service,Homemaker,0.00,0,1.20,Full,0,3,16.60,82.70,49.60,60.80,C,0
4693,STU100694,2022-2023,2025-03-16,11,Male,6,Government-Aided,Urban,4,2,0,Lower-Middle,NaN,Higher Secondary,Primary,Clerical / Service,Homemaker,0.00,0,1.20,Full,0,3,16.60,82.70,49.60,60.80,C,0


**¿Qué vemos?** Hay **25 filas repetidas**: el mismo estudiante aparece dos veces con exactamente los mismos datos. Seguramente se ingresaron dos veces por error. Podemos borrar las copias sin perder información.

## 7. Inconsistencias

Buscamos datos que estén mal escritos o que no tengan sentido.

### 7.1 Columnas de texto

In [11]:
cat_cols = df.select_dtypes(include="object").columns.drop(["student_id", "survey_date"])
for col in cat_cols:
    valores = df[col].dropna()
    espacios = (valores != valores.str.strip()).sum()                           # espacios de más
    variantes = valores.str.strip().str.lower().nunique() != valores.nunique()  # p. ej. "male" vs "Male"
    print(f"{col:<26} únicos={valores.nunique():<3} espacios_extra={espacios:<3} variantes_mayúsc={variantes}")
    print(f"{'':<26} {sorted(valores.unique())}")

academic_year              únicos=4   espacios_extra=0   variantes_mayúsc=False
                           ['2021-2022', '2022-2023', '2023-2024', '2024-2025']
gender                     únicos=3   espacios_extra=0   variantes_mayúsc=False
                           ['Female', 'Male', 'Other']
school_type                únicos=3   espacios_extra=0   variantes_mayúsc=False
                           ['Government', 'Government-Aided', 'Private-Unaided']
residence_type             únicos=3   espacios_extra=0   variantes_mayúsc=False
                           ['Rural', 'Semi-Urban', 'Urban']
family_income_bracket      únicos=5   espacios_extra=0   variantes_mayúsc=False
                           ['High', 'Low', 'Lower-Middle', 'Middle', 'Upper-Middle']
father_education_level     únicos=7   espacios_extra=0   variantes_mayúsc=False


                           ["Bachelor's Degree", 'Doctorate', 'Higher Secondary', "Master's Degree", 'No Formal Education', 'Primary', 'Secondary']
mother_education_level     únicos=7   espacios_extra=0   variantes_mayúsc=False
                           ["Bachelor's Degree", 'Doctorate', 'Higher Secondary', "Master's Degree", 'No Formal Education', 'Primary', 'Secondary']
father_occupation          únicos=7   espacios_extra=0   variantes_mayúsc=False
                           ['Business / Self-Employed', 'Clerical / Service', 'Managerial / Executive', 'Professional', 'Skilled Labor / Trade', 'Unemployed', 'Unskilled Labor']
mother_occupation          únicos=8   espacios_extra=0   variantes_mayúsc=False
                           ['Business / Self-Employed', 'Clerical / Service', 'Homemaker', 'Managerial / Executive', 'Professional', 'Skilled Labor / Trade', 'Unemployed', 'Unskilled Labor']
scholarship_status         únicos=3   espacios_extra=0   variantes_mayúsc=False
               

**¿Qué vemos?** Las categorías están bien escritas. No hay espacios de más ni la misma palabra escrita de dos formas (como "male" y "Male"). En `gender` aparece la opción **"Other"** (349 estudiantes). Es una respuesta válida, así que la dejamos.

### 7.2 Valores fuera de rango

Revisamos que cada número esté dentro de lo lógico. Por ejemplo, un porcentaje no puede pasar de 100 ni una edad ser negativa.

In [12]:
# Las columnas binarias solo deberían tener 0 y 1
binarias = ["single_parent_household", "has_internet_access", "has_computer_or_laptop", "extra_tutoring", "at_risk_flag"]
for col in binarias:
    print(f"{col:<26} valores: {sorted(df[col].dropna().unique().tolist())}")

# Rango lógico (mínimo, máximo) de cada columna numérica
reglas_rango = {
    "age": (5, 25), "grade_level": (1, 12), "family_size": (1, 20), "num_siblings": (0, 15),
    "annual_family_income_inr": (0, None), "distance_to_school_km": (0, 100),
    "parental_involvement_score": (1, 5), "study_hours_per_week": (0, 168),
    "attendance_rate_pct": (0, 100), "previous_year_grade_pct": (0, 100), "final_grade_pct": (0, 100),
}
fuera = []
for col, (lo, hi) in reglas_rango.items():
    s = df[col].dropna()
    n = ((s < lo) | ((s > hi) if hi is not None else False)).sum()
    fuera.append({"variable": col, "min": s.min(), "max": s.max(), "regla": f"[{lo}, {hi if hi is not None else '∞'}]", "fuera_de_rango": n})
pd.DataFrame(fuera)

single_parent_household    valores: [0, 1]
has_internet_access        valores: [0.0, 1.0]
has_computer_or_laptop     valores: [0, 1]
extra_tutoring             valores: [0, 1]
at_risk_flag               valores: [0, 1]


,variable,min,max,regla,fuera_de_rango
0,age,10.00,18.00,"[5, 25]",0
1,grade_level,6.00,12.00,"[1, 12]",0
2,family_size,2.00,10.00,"[1, 20]",0
3,num_siblings,0.00,8.00,"[0, 15]",0
4,annual_family_income_inr,"60,010.00","19,621,849.00","[0, ∞]",0
5,distance_to_school_km,0.20,31.20,"[0, 100]",0
6,parental_involvement_score,1.00,5.00,"[1, 5]",0
7,study_hours_per_week,0.00,65.00,"[0, 168]",0
8,attendance_rate_pct,57.80,100.00,"[0, 100]",0
9,previous_year_grade_pct,23.50,100.00,"[0, 100]",0


**¿Qué vemos?** Todo está en orden. Las columnas binarias solo tienen 0 y 1, y **ningún número se sale de su rango** (no hay porcentajes mayores que 100, edades raras ni distancias negativas).

### 7.3 ¿Las columnas relacionadas coinciden entre sí?

In [13]:
# a) ¿La edad corresponde al grado escolar?
print("a) Edad vs grade_level:")
display(pd.crosstab(df["age"], df["grade_level"]))

a) Edad vs grade_level:


grade_level,6,7,8,9,10,11,12
age,,,,,,,
10,541,0,0,0,0,0,0
11,592,577,0,0,0,0,0
12,566,601,571,0,0,0,0
13,0,563,567,623,0,0,0
14,0,0,579,595,578,0,0
15,0,0,0,583,537,584,0
16,0,0,0,0,558,550,531
17,0,0,0,0,0,602,560
18,0,0,0,0,0,0,567


In [14]:
# b) ¿La nota en letra corresponde a la nota en porcentaje?
print("b) Nota mínima y máxima de cada letra:")
display(df.groupby("grade_class")["final_grade_pct"].agg(["min", "max", "count"]))

# c) ¿Los estudiantes "en riesgo" son los que tienen F?
print("c) at_risk_flag vs grade_class:")
display(pd.crosstab(df["grade_class"], df["at_risk_flag"]))

b) Nota mínima y máxima de cada letra:


,min,max,count
grade_class,,,
A,90.00,100.00,212
B,75.00,89.90,2689
C,60.00,74.90,6161
D,40.00,59.90,2744
F,2.80,39.90,219


c) at_risk_flag vs grade_class:


at_risk_flag,0,1
grade_class,,
A,212,0
B,2689,0
C,6161,0
D,2744,0
F,0,219


In [15]:
# d) ¿El ingreso en dinero coincide con la categoría de ingreso?
print("d) Ingreso anual (INR) por categoría de ingreso:")
orden_ingreso = ["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"]
display(df.groupby("family_income_bracket")["annual_family_income_inr"]
          .agg(["min", "median", "max"]).reindex(orden_ingreso))

d) Ingreso anual (INR) por categoría de ingreso:


,min,median,max
family_income_bracket,,,
Low,"60,010.00","103,053.00","739,228.00"
Lower-Middle,"180,035.00","252,931.00","1,786,799.00"
Middle,"350,014.00","495,187.00","3,478,518.00"
Upper-Middle,"700,918.00","1,051,792.50","7,402,816.00"
High,"1,503,960.00","3,056,531.00","19,621,849.00"


In [16]:
# e) ¿El tamaño de la familia tiene sentido con el número de hermanos?
print("e) family_size - num_siblings, según si es hogar de un solo padre o madre:")
display(pd.crosstab(df["family_size"] - df["num_siblings"], df["single_parent_household"],
                    rownames=["family_size - num_siblings"]))
print("Casos con más hermanos que miembros de la familia:", (df["num_siblings"] >= df["family_size"]).sum())

e) family_size - num_siblings, según si es hogar de un solo padre o madre:


single_parent_household,0,1
family_size - num_siblings,,
2,5492,908
3,4849,776


Casos con más hermanos que miembros de la familia: 0


In [17]:
# f) ¿La fecha de la encuesta coincide con el año académico?
fechas = pd.to_datetime(df["survey_date"], errors="coerce")
print("f) Fechas que no se pudieron leer:", fechas.isna().sum())
print("   Primera y última fecha:", fechas.min().date(), "a", fechas.max().date())
display(df["academic_year"].value_counts().sort_index())

f) Fechas que no se pudieron leer: 0


   Primera y última fecha: 2025-03-01 a 2025-05-14


academic_year
2021-2022    2127
2022-2023    2713
2023-2024    3345
2024-2025    3840
Name: count, dtype: int64

**¿Qué encontramos?**

| Revisión | Resultado | ¿Qué hacemos? |
|---|---|---|
| a) Edad vs. grado | Bien: cada grado tiene estudiantes de 3 edades seguidas (por ejemplo, 6.º grado tiene de 10 a 12 años). | Nada. |
| b) Letra vs. nota | Bien: A = 90 o más, B = 75-89.9, C = 60-74.9, D = 40-59.9, F = menos de 40. | Nada. La letra sale directamente de la nota. |
| c) Riesgo vs. letra | `at_risk_flag` vale 1 solo cuando la nota es F (219 estudiantes). | Nada, pero ambas columnas dicen lo mismo. Conviene **no usar las dos** en K-Means. |
| d) Ingreso vs. categoría | Los rangos **se mezclan**: hay hogares "Low" que ganan más que algunos "Middle". La mediana sí va subiendo en orden. | Los dejamos así, porque la categoría puede considerar otras cosas además del ingreso. Lo anotamos como posible inconsistencia. |
| e) Familia vs. hermanos | No hay casos imposibles. La diferencia entre miembros y hermanos siempre es 2 o 3, sin relación clara con si es hogar de un solo padre o madre. | Los dejamos así. No sabemos exactamente cómo se contó `family_size`. |
| f) Fecha vs. año | Todas las encuestas son de **marzo a mayo de 2025**, aunque haya datos de años anteriores (2021-2022, por ejemplo). | Pensamos que la fecha es cuándo se recogieron los datos. No la usamos en el análisis. |

## 8. Valores atípicos

Un valor atípico es un dato muy alejado del resto. Para encontrarlos usamos el **método IQR**:
1. Calculamos Q1 (el 25% de los datos está por debajo) y Q3 (el 75% está por debajo).
2. Calculamos IQR = Q3 − Q1.
3. Un dato es atípico si es menor que Q1 − 1.5·IQR o mayor que Q3 + 1.5·IQR.

Aquí **solo los contamos**. Los gráficos de atípicos (boxplots) se hacen en la parte 03.

In [18]:
num_cols = ["age", "family_size", "num_siblings", "annual_family_income_inr", "distance_to_school_km",
            "parental_involvement_score", "study_hours_per_week", "attendance_rate_pct",
            "previous_year_grade_pct", "final_grade_pct"]
resumen_atipicos = []
for col in num_cols:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr     # límites para considerar algo "normal"
    n = ((s < lim_inf) | (s > lim_sup)).sum()             # cuántos se salen de esos límites
    resumen_atipicos.append({"variable": col, "lim_inf": lim_inf, "lim_sup": lim_sup,
                             "min": s.min(), "max": s.max(), "atipicos": n, "%": round(n / len(s) * 100, 2)})
pd.DataFrame(resumen_atipicos).sort_values("atipicos", ascending=False)

,variable,lim_inf,lim_sup,min,max,atipicos,%
3,annual_family_income_inr,"-499,442.25","1,344,747.75","60,010.00","19,621,849.00",1131,10.01
4,distance_to_school_km,-3.90,10.50,0.20,31.20,692,5.87
5,parental_involvement_score,1.50,5.50,1.00,5.00,340,2.83
9,final_grade_pct,38.35,96.35,2.80,100.00,240,2.00
6,study_hours_per_week,2.30,26.30,0.00,65.00,122,1.01
8,previous_year_grade_pct,33.65,90.85,23.50,100.00,80,0.69
7,attendance_rate_pct,62.95,108.95,57.80,100.00,21,0.18
2,num_siblings,-2.00,6.00,0.00,8.00,20,0.17
1,family_size,1.00,9.00,2.00,10.00,5,0.04
0,age,6.00,22.00,10.00,18.00,0,0.00


**¿Qué vemos?**
- **Ingreso familiar:** es la columna con más atípicos (cerca del 10%). La mayoría de las familias gana poco y unas pocas ganan muchísimo (hasta 19.6 millones de INR). Son familias de ingreso "High", no errores.
- **Distancia a la escuela:** pasa algo parecido. Casi todos viven a menos de 10 km, pero algunos (sobre todo en zonas rurales) viven hasta a 31 km.
- **Participación de los padres:** los "atípicos" son simplemente los que tienen 1 en una escala del 1 al 5. Es un valor válido; el método IQR no funciona tan bien con escalas cortas.
- **Horas de estudio y nota final:** algunos estudian mucho (hasta 65 horas por semana) y otros tienen notas muy bajas (2.8%). Son casos extremos, pero posibles.

**Decisión:** **no borramos** los atípicos, porque no son errores. En la parte 04 se puede reducir su efecto escalando los datos o aplicando logaritmo al ingreso.

## 9. Limpieza de datos

### 9.1 Borrar duplicados

In [19]:
antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)   # borra las filas repetidas
print(f"Registros antes: {antes:,} | después: {len(df):,} | eliminados: {antes - len(df)}")
assert df["student_id"].is_unique   # comprobamos que ya no hay códigos repetidos

Registros antes: 12,025 | después: 12,000 | eliminados: 25

### 9.2 Rellenar valores vacíos

En lugar de rellenar todos los vacíos con un solo valor general, los rellenamos **según el grupo** al que pertenece cada estudiante. Así el valor que ponemos es más realista. Por ejemplo, si a una familia de ingreso "Low" le falta el ingreso, le ponemos el ingreso típico de las familias "Low", no el de todo el dataset.

- **Mediana:** el valor del centro. La usamos para números porque no se ve afectada por los valores extremos.
- **Moda:** el valor que más se repite. La usamos para texto y para 0/1.

| Columna(s) | Cómo la rellenamos | ¿Por qué? |
|---|---|---|
| `annual_family_income_inr` | Mediana según la categoría de ingreso | La categoría de ingreso no tiene vacíos y nos dice el nivel económico de la familia. |
| `has_internet_access` | Moda según la categoría de ingreso | Tener internet depende mucho del ingreso (35% en "Low" y 72% en "High"). |
| Educación y trabajo del padre y la madre | Moda según la categoría de ingreso | También cambian según el nivel económico. |
| `distance_to_school_km` | Mediana según el tipo de residencia | En zona rural viven más lejos (5.4 km) que en zona urbana (1.7 km). |
| `attendance_rate_pct`, `previous_year_grade_pct` | Mediana según la nota en letra | Las dos tienen que ver con el rendimiento del estudiante. |

In [20]:
# Comprobamos que el acceso a internet sí depende del ingreso (proporción por categoría)
pd.crosstab(df["family_income_bracket"], df["has_internet_access"], normalize="index").reindex(orden_ingreso).round(2)

has_internet_access,0.00,1.00
family_income_bracket,,
Low,0.65,0.35
Lower-Middle,0.54,0.46
Middle,0.46,0.54
Upper-Middle,0.41,0.59
High,0.28,0.72


In [21]:
nulos_antes = df.isna().sum()

# Función que devuelve el valor más repetido (la moda)
def moda(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan

# Columnas numéricas -> mediana de su grupo
df["annual_family_income_inr"] = df["annual_family_income_inr"].fillna(
    df.groupby("family_income_bracket")["annual_family_income_inr"].transform("median"))
df["distance_to_school_km"] = df["distance_to_school_km"].fillna(
    df.groupby("residence_type")["distance_to_school_km"].transform("median"))
for col in ["attendance_rate_pct", "previous_year_grade_pct"]:
    df[col] = df[col].fillna(df.groupby("grade_class")[col].transform("median"))

# Columnas de texto y de 0/1 -> moda de su grupo
for col in ["has_internet_access", "father_education_level", "mother_education_level",
            "father_occupation", "mother_occupation"]:
    df[col] = df[col].fillna(df.groupby("family_income_bracket")[col].transform(moda))

# Comparamos los vacíos antes y después
pd.DataFrame({"nulos_antes": nulos_antes, "nulos_despues": df.isna().sum()}).query("nulos_antes > 0")

,nulos_antes,nulos_despues
annual_family_income_inr,720,0
father_education_level,420,0
mother_education_level,360,0
father_occupation,480,0
mother_occupation,600,0
has_internet_access,180,0
distance_to_school_km,240,0
attendance_rate_pct,300,0
previous_year_grade_pct,360,0


### 9.3 Corregir tipos de datos

In [22]:
# Ya no tiene vacíos, así que la pasamos de decimal a entero (0/1)
df["has_internet_access"] = df["has_internet_access"].astype(int)

# Pasamos la fecha de texto a tipo fecha
df["survey_date"] = pd.to_datetime(df["survey_date"], format="%Y-%m-%d")

# Columnas de texto que tienen un orden lógico (de menor a mayor)
ordenes = {
    "academic_year": ["2021-2022", "2022-2023", "2023-2024", "2024-2025"],
    "family_income_bracket": orden_ingreso,
    "father_education_level": ["No Formal Education", "Primary", "Secondary", "Higher Secondary",
                               "Bachelor's Degree", "Master's Degree", "Doctorate"],
    "scholarship_status": ["No Scholarship", "Partial", "Full"],
    "grade_class": ["F", "D", "C", "B", "A"],
}
ordenes["mother_education_level"] = ordenes["father_education_level"]
for col, orden in ordenes.items():
    assert set(df[col].unique()) <= set(orden), col   # verificamos que no falte ninguna categoría
    df[col] = pd.Categorical(df[col], categories=orden, ordered=True)

# Columnas de texto sin orden
for col in ["gender", "school_type", "residence_type", "father_occupation", "mother_occupation"]:
    df[col] = df[col].astype("category")

df.dtypes.to_frame("tipo_final")

,tipo_final
student_id,object
academic_year,category
survey_date,datetime64[ns]
age,int64
gender,category
grade_level,int64
school_type,category
residence_type,category
family_size,int64
num_siblings,int64


**Nota:** al guardar en CSV se pierde el tipo `category` y el orden de las categorías. Si alguien lo necesita en otra parte (por ejemplo, en la parte 04), puede copiar el diccionario `ordenes` de este notebook.

## 10. Revisión final y exportación

In [23]:
print(f"Dimensiones finales: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Vacíos restantes:    {df.isna().sum().sum()}")
print(f"Duplicados:          {df.duplicated().sum()}")
print(f"IDs únicos:          {df['student_id'].is_unique}")

# Si algo de esto falla, el notebook muestra un error
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0
df.describe().T

Dimensiones finales: 12,000 filas x 29 columnas
Vacíos restantes:    0
Duplicados:          0
IDs únicos:          True


,count,mean,min,25%,50%,75%,max,std
survey_date,12000,2025-04-07 00:34:12,2025-03-01 00:00:00,2025-03-19 00:00:00,2025-04-07 00:00:00,2025-04-26 00:00:00,2025-05-14 00:00:00,NaN
age,"12,000.00",13.99,10.00,12.00,14.00,16.00,18.00,2.15
grade_level,"12,000.00",8.98,6.00,7.00,9.00,11.00,12.00,1.99
family_size,"12,000.00",4.60,2.00,4.00,5.00,6.00,10.00,1.40
num_siblings,"12,000.00",2.13,0.00,1.00,2.00,3.00,8.00,1.42
single_parent_household,"12,000.00",0.14,0.00,0.00,0.00,0.00,1.00,0.35
annual_family_income_inr,"12,000.00","643,321.01","60,010.00","193,876.50","351,958.00","653,176.50","19,621,849.00","919,962.99"
has_internet_access,"12,000.00",0.50,0.00,0.00,0.00,1.00,1.00,0.50
has_computer_or_laptop,"12,000.00",0.42,0.00,0.00,0.00,1.00,1.00,0.49
distance_to_school_km,"12,000.00",3.87,0.20,1.50,2.80,5.10,31.20,3.55


In [24]:
# Comparación entre el dataset original y el limpio
resumen = pd.DataFrame({
    "Métrica": ["Filas", "Columnas", "Celdas vacías", "Filas duplicadas"],
    "Original": [df_raw.shape[0], df_raw.shape[1], df_raw.isna().sum().sum(), df_raw.duplicated().sum()],
    "Limpio": [df.shape[0], df.shape[1], df.isna().sum().sum(), df.duplicated().sum()],
})
resumen

,Métrica,Original,Limpio
0,Filas,12025,12000
1,Columnas,29,29
2,Celdas vacías,3667,0
3,Filas duplicadas,25,0


In [25]:
# Guardamos el dataset limpio
RUTA_PROCESSED.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_PROCESSED, index=False, encoding="utf-8")
print(f"Dataset limpio guardado en: {RUTA_PROCESSED} ({len(df):,} registros)")

# Lo volvemos a leer para comprobar que se guardó bien
pd.read_csv(RUTA_PROCESSED).shape

Dataset limpio guardado en: ..\..\data\processed\calificaciones_limpio.csv (12,000 registros)


(12000, 29)

## 11. Conclusiones

1. **Cómo es el dataset:** tiene **12,025 filas y 29 columnas** con información de cada estudiante: datos personales, familia, situación económica, hábitos de estudio y notas. Hay 11 columnas categóricas, 11 numéricas, 5 binarias (0/1), 1 código de estudiante y 1 fecha.
2. **Calidad de los datos:** en general vienen bien. No hay palabras mal escritas ni números imposibles. Además, las columnas relacionadas coinciden: la edad corresponde al grado, la letra a la nota y "en riesgo" a la F.
3. **Duplicados:** encontramos **25 filas repetidas** y las borramos. Quedan **12,000 estudiantes**.
4. **Datos vacíos:** había **3,667 celdas vacías en 9 columnas**, ninguna con más del 6%, pero repartidas en el 26.8% de las filas. Por eso no borramos esas filas y las **rellenamos según el grupo** de cada estudiante: mediana para los números y moda para el texto. Al final no queda ningún vacío.
5. **Tipos de datos:** pasamos `has_internet_access` a entero y `survey_date` a fecha. A las columnas de texto les dimos tipo categoría, con orden en las que lo tienen (ingreso, educación, beca, nota en letra y año).
6. **Cosas raras que anotamos pero no cambiamos:**
   - Los rangos de ingreso se mezclan entre categorías.
   - La fecha de la encuesta no coincide con el año académico.
   - No está claro cómo se contó el tamaño de la familia.

   No hacen que los datos estén mal, pero hay que tenerlas en cuenta al sacar conclusiones.
7. **Recomendaciones para los compañeros:**
   - **Atípicos:** no los borramos. El ingreso tiene unos pocos valores muy altos, así que conviene aplicarle logaritmo o un escalado antes de K-Means.
   - **Columnas repetidas:** `grade_class` y `at_risk_flag` salen de `final_grade_pct`, así que no conviene usarlas juntas en K-Means.
   - **Columnas que no sirven para el modelo:** `student_id` y `survey_date` no ayudan a formar grupos.

**Archivo final:** `data/processed/calificaciones_limpio.csv` — 12,000 filas x 29 columnas, sin vacíos ni duplicados.